In [2]:
import os # type: ignore
import numpy as np # type: ignore
import h5py # type: ignore
import json # type: ignore
import torch # type: ignore
import matplotlib.pyplot as plt # type: ignore
import torch.nn as nn # type: ignore
import torchvision # type: ignore
from tqdm import tqdm # type: ignore
from collections import Counter # type: ignore
from random import seed, choice, sample # type: ignore
from torch.utils.data import Dataset # type: ignore
from PIL import Image # type: ignore
import  torchvision.transforms as transforms # type: ignore
from torch.nn.utils.rnn import pack_padded_sequence # type: ignore
import torch_directml # type: ignore

In [3]:
# Init device
# device = torch_directml.device(0)
# print(torch_directml.device_name(0))
device = 'cpu'

In [4]:
# Save checkpoint
def save_checkpoint(epoch, encoder, decoder, decoder_optimizer):
    state = {
        'epoch' : epoch,
        'encoder' : encoder,
        'decoder' : decoder,
        'decoder_optimizer' : decoder_optimizer
    }
    filename = os.path.join(f'./source/checkpoint{epoch}.pth')
    torch.save(state, filename)

In [5]:
# Average mater
class AverageMeter:
    def __init__(self):
        self.value = 0.0
        self.average = 0.0
        self.sum = 0.0
        self.count = 0
    def update(self, value, n=1):
        self.value = value
        self.sum += value * n
        self.count += n
        self.average = self.sum / self.count

In [6]:
# Adjust learning rate
def adjust_learning_rate(optimizer, shrink_factor):
    optimizer.param_groups[0]['lr'] = optimizer.param_groups[0]['lr'] * shrink_factor
    print(f"New learning rate: {optimizer.param_groups[0]['lr']:.3f}")

In [7]:
# Calculate top-k accuracy
def accuracy(scores, targets, k=1):
    batch_size = targets.size(0)
    _, ind = scores.topk(k, 1)
    correct = (ind == targets.view(-1, 1).expand_as(ind))
    correct_total = correct.sum()
    return 100.0 * correct_total / batch_size

In [8]:
# Create data files
def create_input_files(karpathy_json_path, image_folder, captions_per_image,
                       min_word_freq, output_folder, max_len=100):
    with open(karpathy_json_path, 'r') as j:
        data = json.load(j)

    train_image_paths = []
    train_image_captions = []
    val_image_paths = []
    val_image_captions = []
    test_image_paths = []
    test_image_captions = []
    word_freq = Counter()

    for img in data['images']:
        captions = []
        for c in img['sentences']:
            word_freq.update(c['tokens'])
            if len(c['tokens']) <= max_len:
                captions.append(c['tokens'])

        if len(captions) == 0:
            continue

        path = os.path.join(image_folder, img['filename'])

        if img['split'] in {'train', 'restval'}:
            train_image_paths.append(path)
            train_image_captions.append(captions)
        elif img['split'] in {'val'}:
            val_image_paths.append(path)
            val_image_captions.append(captions)
        elif img['split'] in {'test'}:
            test_image_paths.append(path)
            test_image_captions.append(captions)

    words = [w for w in word_freq.keys() if word_freq[w] > min_word_freq]
    word_map = {k: v + 1 for v, k in enumerate(words)}
    word_map['<unk>'] = len(word_map) + 1
    word_map['<start>'] = len(word_map) + 1
    word_map['<end>'] = len(word_map) + 1
    word_map['<pad>'] = 0

    with open(os.path.join(output_folder, 'WORDMAP' + '.json'), 'w') as j:
        json.dump(word_map, j)

    seed(123)
    for impaths, imcaps, split in [(train_image_paths, train_image_captions, 'TRAIN'),
                                   (val_image_paths, val_image_captions, 'VAL'),
                                   (test_image_paths, test_image_captions, 'TEST')]:

        with h5py.File(os.path.join(output_folder, split + '_IMAGES' + '.hdf5'), 'a') as h:
            h.attrs['captions_per_image'] = captions_per_image
            images = h.create_dataset('images', (len(impaths), 3, 256, 256), dtype='uint8')
            print("\nReading %s images and captions, storing to file...\n" % split)
            enc_captions = []
            caplens = []
            for i, path in enumerate(tqdm(impaths)):
                if len(imcaps[i]) < captions_per_image:
                    captions = imcaps[i] + [choice(imcaps[i]) for _ in range(captions_per_image - len(imcaps[i]))]
                else:
                    captions = sample(imcaps[i], k=captions_per_image)
                img = Image.open(impaths[i])
                img = img.resize((256, 256))
                img = np.array(img)
                if len(img.shape) == 2:
                    img = img[:, :, np.newaxis]
                    img = np.concatenate([img, img, img], axis=2)

                img = img.transpose(2, 0, 1)
                images[i] = img

                for j, c in enumerate(captions):
                    enc_c = [word_map['<start>']] + [word_map.get(word, word_map['<unk>']) for word in c] + [
                        word_map['<end>']] + [word_map['<pad>']] * (max_len - len(c))
                    c_len = len(c) + 2
                    enc_captions.append(enc_c)
                    caplens.append(c_len)

            with open(os.path.join(output_folder, split + '_CAPTIONS' + '.json'), 'w') as j:
                json.dump(enc_captions, j)
            with open(os.path.join(output_folder, split + '_CAPLENS' + '.json'), 'w') as j:
                json.dump(caplens, j)

In [68]:
# create_input_files('./source/archive/dataset_flickr8k.json',
#                    './source/archive/images',
#                    5,
#                    2,
#                    './source/archive/flickr8k',
#                    30)

In [9]:
# Dataset class
base_path = './source/archive/flickr8k'
class Flickr8kDataset(Dataset):
    def __init__(self, image_transformer):
        self.h = h5py.File(os.path.join(base_path, 'TRAIN_IMAGES.hdf5'), 'r')
        self.imgs = self.h['images']
        self.cpi = self.h.attrs['captions_per_image']
        with open(os.path.join(base_path, 'TRAIN_CAPTIONS.json'), 'r') as j:
            self.captions = json.load(j)
        with open(os.path.join(base_path, 'TRAIN_CAPLENS.json'), 'r') as j:
            self.caplens = json.load(j)
        self.image_transformer = image_transformer
        self.dataset_size = len(self.captions)
    def __len__(self):
        return self.dataset_size
    def __getitem__(self, i):
        img = self.imgs[i // self.cpi]
        img = torch.FloatTensor(img)
        img = img / 255
        if self.image_transformer:
            img = self.image_transformer(img)
        caption = self.captions[i]
        caption = torch.LongTensor(caption)
        caplen = self.caplens[i]
        caplen = torch.LongTensor([caplen])
        return img, caption, caplen

In [10]:
# Visualize data
vis_train_loader = torch.utils.data.DataLoader(Flickr8kDataset(None),
                                           batch_size=10,
                                           shuffle=True,
                                           pin_memory=True)
vis_img, vis_caption, vis_caplen = next(iter(vis_train_loader))
print(vis_img.shape)
print(vis_caption.shape)
print(vis_caplen.shape)

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = './source/archive/flickr8k\TRAIN_IMAGES.hdf5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [17]:
# Creating encoder class
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = torchvision.models.resnet101(pretrained=True)
        all_modules = list(resnet.children())
        modules = all_modules[:-2]
        self.resnet = nn.Sequential(*modules)
        self.avgpool = nn.AvgPool2d(8)
        self.fine_tune(False)
    def fine_tune(self, fine_tune):
        for p in self.resnet.parameters():
            p.requires_grad_(fine_tune)
    def forward(self, images):
        batch_size = images.shape[0]
        encoded_images = self.resnet(images)
        global_features = self.avgpool(encoded_images)
        global_features = global_features.view(batch_size, -1)
        return global_features

In [12]:
# Visualize data
vis_encoder = Encoder()
vis_img = torch.randn(10, 3, 256, 256)
vis_out = vis_encoder(vis_img)
print(vis_out.shape)

C:\Alexey\Projects\Udemy\NN_learning\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Alexey\Projects\Udemy\NN_learning\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet101_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet101_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


torch.Size([10, 2048])


In [13]:
# Decoder class
class Decoder(nn.Module):
    def __init__(self, embed_dim, decoder_dim, vocabulary_size, encoder_dim=2048):
        super().__init__()
        self.embed_dim = embed_dim
        self.decoder_dim = decoder_dim
        self.vocabulary_size = vocabulary_size
        self.encoder_dim = encoder_dim
        self.embedding = nn.Embedding(vocabulary_size, embed_dim)
        self.lstm = nn.LSTMCell(embed_dim + encoder_dim, decoder_dim)
        self.fc = nn.Linear(decoder_dim, vocabulary_size)
        self.init_weights()
    def init_weights(self):
        self.embedding.weight.data.uniform_(-0.1, 0.1)
        self.fc.weight.data.uniform_(-0.1, 0.1)
        self.fc.bias.data.fill_(0)
    def init_hidden(self, batch_size):
        h = torch.zeros(batch_size, self.decoder_dim).to(device)
        c = torch.zeros(batch_size, self.decoder_dim).to(device)
        return h, c
    def forward(self, global_image, encoded_captions, caption_lengths):
        batch_size = global_image.size(0)
        caption_lengths, sort_ind = caption_lengths.squeeze(1).sort(dim=0,
                                                                    descending=True)
        global_image = global_image[sort_ind]
        encoded_captions = encoded_captions[sort_ind]
        embeddings = self.embedding(encoded_captions)
        h, c = self.init_hidden(batch_size)
        decode_lengths = (caption_lengths - 1).tolist()
        predictions = torch.zeros((batch_size, max(decode_lengths), self.vocabulary_size)).to(device)
        for t in range(max(decode_lengths)):
            batch_size_t = sum([l > t for l in decode_lengths])
            lstm_input = torch.cat([embeddings[:batch_size_t, t, :], global_image[:batch_size_t]], dim=-1)
            h, c = self.lstm(lstm_input, (h[:batch_size_t], c[:batch_size_t]))
            preds = self.fc(h)
            predictions[:batch_size_t, t, :] = preds
        return predictions, encoded_captions, decode_lengths, sort_ind

In [74]:
# Train function
def train(data_loader, encoder, decoder, criterion, decoder_optimizer, epoch):
    encoder.train()
    decoder.train()
    losses = AverageMeter()
    top_3_accs = AverageMeter()
    for i, (img, caption, caplen) in enumerate(data_loader):
        img = img.to(device)
        caption = caption.to(device)
        caplen = caplen.to(device)
        global_features = encoder(img)
        scores, caps_sorted, decoded_lengths, sort_indexes = decoder(global_features, caption, caplen)
        targets = caps_sorted[:, 1:]
        scores = pack_padded_sequence(scores, decoded_lengths, batch_first=True)
        targets = pack_padded_sequence(targets, decoded_lengths, batch_first=True)
        loss = criterion(scores.data, targets.data)
        decoder_optimizer.zero_grad()
        loss.backward()
        decoder_optimizer.step()
        top_3_accs.update(accuracy(scores.data, targets.data, 3), sum(decoded_lengths))
        losses.update(loss.item(), sum(decoded_lengths))

        if i % 100 == 0:
            print(f"Epoch: {epoch} ({i}/{len(train_loader)})")
            print(f"Loss: {losses.value:.4f} ({losses.average:.4f})")
            print(f"Top-3 Accuracy: {top_3_accs.value:.4f} ({top_3_accs.average:.4f})")

In [22]:
# Define hyperparams
embed_dim = 512
decoder_dim = 512
decoder_learning_rate = 5e-4
start_epoch = 0
epochs = 20
batch_size = 60
checkpoint : str | None | dict = './source/archive/checkpoint9.pth'
with open('./source/archive/flickr8k/WORDMAP.json', 'r') as j:
    word_map = json.load(j)
encoder = Encoder()
decoder = Decoder(embed_dim, decoder_dim, len(word_map))
rev_word_map = {val : key for key, val in word_map.items()}
if checkpoint is None:
    encoder = Encoder()
    decoder = Decoder(embed_dim, decoder_dim, len(word_map))
    decoder_optimizer = torch.optim.Adam(decoder.parameters(),
                                        lr=decoder_learning_rate)
else:
    checkpoint = torch.load(checkpoint)
    start_epoch = checkpoint['epoch'] + 1
    encoder = checkpoint['encoder']
    decoder = checkpoint['decoder']
    decoder_optimizer = checkpoint['decoder_optimizer']
encoer = encoder.to(device)
decoder = decoder.to(device)
criterion = nn.CrossEntropyLoss().to(device)
normalization = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
transformer = transforms.Compose([normalization])
train_loader = torch.utils.data.DataLoader(Flickr8kDataset(transformer),
                                           batch_size=batch_size,
                                           shuffle=True,
                                           pin_memory=True)

FileNotFoundError: [Errno 2] No such file or directory: './source/archive/flickr8k/WORDMAP.json'

In [14]:
# Evaluation function
def greedy_decode(image):
    max_len = 20
    encoder.eval()
    decoder.eval()
    sampled = []
    img = Image.open(image)
    plt.axis('off')
    plt.imshow(img)
    img = img.resize((256, 256))
    img = np.array(img)
    img = img.transpose(2, 0, 1)
    img = img / 255.0
    img = torch.FloatTensor(img).to(device)
    img = transformer(img)
    img = img.unsqueeze(0)

    global_features = encoder(img)
    pred = torch.LongTensor([[word_map['<start>']]])
    h, c = decoder.init_hidden(1)
    for t in range(max_len):
        embeddings = decoder.embedding(pred).squeeze(1)
        lstm_input = torch.cat([embeddings, global_features], dim=-1)
        h, c = decoder.lstm(lstm_input, (h, c))
        preds = decoder.fc(h)
        _, pred = preds.max(dim=1)
        sampled.append(pred)
        if pred == word_map['<end>']:
            break
    generated_words = [rev_word_map[sampled[i].item()] for i in range(len(sampled))]
    return ' '.join(generated_words)

In [15]:
# Training the model
# for epoch in range(start_epoch, epochs):
#     if epochs % 3 == 0 and epoch != 0:
#         adjust_learning_rate(decoder_optimizer, 0.8)
#     train(train_loader, encoder, decoder, criterion, decoder_optimizer, epoch)
#     save_checkpoint(epoch, encoder, decoder, decoder_optimizer)

In [21]:
greedy_decode('./source/images/car.jpg')

NameError: name 'decoder' is not defined